# Preprocessing
Cleans raw resume text and derives one representative job description per category
(the dataset ships resumes with a `Category` label but no separate job postings, so this
gives us both sides of the matching problem for later sprint stages).

Assumes `download_data.ipynb` has already been run (so `data/raw/Resume.csv` exists).
Run `config.ipynb` first (or via the `%run` cell below) so the path variables exist.


In [6]:
%run config.ipynb

Config loaded. Data will be stored under: c:\Users\RanVic\Downloads\applicant_screening_sprint2\applicant_screening_nb\data_pipeline\data


In [7]:
%pip install beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\RanVic\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import re
import logging
import pandas as pd
from bs4 import BeautifulSoup
from collections import Counter

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("preprocess")


def strip_html(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return BeautifulSoup(text, "html.parser").get_text(separator=" ")


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = strip_html(text)
    text = re.sub(r"[^\x20-\x7E]", " ", text)
    text = normalize_whitespace(text)
    return text


def _basic_stopwords() -> set:
    """Small built-in stopword list so this stage works before NLTK data is
    downloaded — the full NLTK stopword list is used in the tokenization/TF-IDF stage."""
    return {
        "the", "and", "for", "with", "this", "that", "from", "have", "has",
        "was", "were", "are", "you", "your", "our", "their", "his", "her",
        "company", "name", "city", "state", "using", "including", "within",
    }


def load_raw_resumes() -> pd.DataFrame:
    if not RAW_RESUME_CSV.exists():
        raise FileNotFoundError(
            f"Raw dataset not found at {RAW_RESUME_CSV}. Run download_data.ipynb first."
        )
    df = pd.read_csv(RAW_RESUME_CSV)
    logger.info(f"Loaded {len(df)} raw resumes from {RAW_RESUME_CSV}")
    return df


def build_clean_resumes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if COL_RESUME_TEXT in df.columns:
        df["clean_text"] = df[COL_RESUME_TEXT].apply(clean_text)
    else:
        df["clean_text"] = df[COL_RESUME_HTML].apply(clean_text)

    before = len(df)
    df = df[df["clean_text"].str.len() >= MIN_RESUME_LENGTH_CHARS]
    df = df.drop_duplicates(subset="clean_text")
    after = len(df)
    logger.info(f"Dropped {before - after} rows (too short / duplicate). {after} resumes remain.")

    keep_cols = [c for c in [COL_ID, COL_CATEGORY] if c in df.columns] + ["clean_text"]
    return df[keep_cols].reset_index(drop=True)


def build_job_descriptions(clean_resumes: pd.DataFrame) -> pd.DataFrame:
    stopwords = _basic_stopwords()
    rows = []
    for category, group in clean_resumes.groupby(COL_CATEGORY):
        word_counts = Counter()
        for text in group["clean_text"]:
            words = re.findall(r"[a-zA-Z]{3,}", text.lower())
            word_counts.update(w for w in words if w not in stopwords)
        top_terms = [w for w, _ in word_counts.most_common(40)]
        description = f"Looking for a {category} candidate with experience in: " + ", ".join(top_terms)
        rows.append({COL_CATEGORY: category, "job_description": description})
    logger.info(f"Built {len(rows)} pseudo job descriptions (one per category).")
    return pd.DataFrame(rows)


def run():
    raw = load_raw_resumes()
    resumes_clean = build_clean_resumes(raw)
    resumes_clean.to_csv(PROCESSED_RESUMES_CSV, index=False)
    logger.info(f"Saved cleaned resumes to {PROCESSED_RESUMES_CSV}")

    job_descriptions = build_job_descriptions(resumes_clean)
    job_descriptions.to_csv(JOB_DESCRIPTIONS_CSV, index=False)
    logger.info(f"Saved job descriptions to {JOB_DESCRIPTIONS_CSV}")
    return resumes_clean, job_descriptions


resumes_clean, job_descriptions = run()

2026-09-22 14:50:57,764 | INFO | Loaded 2484 raw resumes from c:\Users\RanVic\Downloads\applicant_screening_sprint2\applicant_screening_nb\data_pipeline\data\raw\Resume.csv
2026-09-22 14:51:03,162 | INFO | Dropped 3 rows (too short / duplicate). 2481 resumes remain.
2026-09-22 14:51:05,546 | INFO | Saved cleaned resumes to c:\Users\RanVic\Downloads\applicant_screening_sprint2\applicant_screening_nb\data_pipeline\data\processed\resumes_clean.csv
2026-09-22 14:51:09,866 | INFO | Built 24 pseudo job descriptions (one per category).
2026-09-22 14:51:09,873 | INFO | Saved job descriptions to c:\Users\RanVic\Downloads\applicant_screening_sprint2\applicant_screening_nb\data_pipeline\data\processed\job_descriptions.csv


### Sanity check

In [9]:
display(resumes_clean.head())
display(job_descriptions.head())

,ID,Category,clean_text
0,16852973,HR,HR ADMINISTRATOR/MARKETING ASSOCIATE HR ADMINI...
1,22323967,HR,"HR SPECIALIST, US HR OPERATIONS Summary Versat..."
2,33176873,HR,HR DIRECTOR Summary Over 20 years experience i...
3,27018550,HR,"HR SPECIALIST Summary Dedicated, Driven, and D..."
4,17812897,HR,HR MANAGER Skill Highlights HR SKILLS HR Depar...


,Category,job_description
0,ACCOUNTANT,Looking for a ACCOUNTANT candidate with experi...
1,ADVOCATE,Looking for a ADVOCATE candidate with experien...
2,AGRICULTURE,Looking for a AGRICULTURE candidate with exper...
3,APPAREL,Looking for a APPAREL candidate with experienc...
4,ARTS,Looking for a ARTS candidate with experience i...
